# Projeto de Aprendizado de Máquina — v4

## Classificador binário para diferenciar:
- **classe = 1** → EIA (Estudos de Impacto Ambiental)
- **classe = 0** → Outros documentos de licenciamento (relatórios, programas, etc.)

## Melhorias em relação à v3:
1. **Ensemble corrigido**: LinearSVC substitui o RandomForest (RF tinha recall de 33% para EIA);
2. **`voting='soft'` com pesos**: modelos contribuem com probabilidades ponderadas em vez de votos binários;
3. **Métricas de validação cruzada focadas em F1**: acurácia é enganosa com classes desbalanceadas;
4. **Threshold tuning**: ajuste do limiar de decisão para maximizar recall de EIA com mínima perda de precisão;
5. **Validação cruzada final antes de salvar**: o modelo de produção é avaliado antes do `joblib.dump`.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_curve, roc_auc_score
)

import joblib
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


## 2. Carregar CSV e inspecionar

In [2]:
caminho_csv = "dataset_eia_vs_outro.csv"

df = pd.read_csv(caminho_csv)

print("Primeiras linhas:")
print(df.head())
print("\nFormato (linhas, colunas):", df.shape)
print("\nDistribuição da classe:")
contagem = df["classe"].value_counts()
print(contagem)
ratio = contagem[0] / contagem[1]
print(f"\nRatio desbalanceamento (0/1): {ratio:.2f}x")
print("→ Modelos precisam de class_weight='balanced' para compensar.")

Primeiras linhas:
                                             arquivo  \
0                  01-0085-06-A-001 EIA FINAL V2.PDF   
1  0643840 - Estudo Ambiental TOMO I e II _064384...   
2              07__reas de Influ_ncia do Projeto.pdf   
3     08_Diagn_stico Ambiental_MF_Parte 01 de 03.pdf   
4                       09 - Medidas e Programas.pdf   

                                               texto  classe  
0  MMX - MINAS - RIO MINERAÇÃO E LOGÍSTICA LTDA. ...       1  
1  DNIT\nC G M A B\nSUMÁRIO\nTOMO I\n1. INTRODUÇÃ...       1  
2  7. ÁREAS DE INFLUÊNCIA DO PROJETO\nA área de i...       1  
3  8. DIAGNÓSTICO AMBIENTAL DA ÁREA DE INFLUÊNCIA...       1  
4  Estudo de Impacto Ambiental Medidas e Programa...       1  

Formato (linhas, colunas): (155, 3)

Distribuição da classe:
classe
0    79
1    76
Name: count, dtype: int64

Ratio desbalanceamento (0/1): 1.04x
→ Modelos precisam de class_weight='balanced' para compensar.


## 3. Configurar NLTK + stopwords

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('rslp')

stopwords_base = set(stopwords.words('portuguese'))

stopwords_extra = {
    # Jurídico/burocrático
    "art", "artigo", "parágrafo", "paragrafo", "inciso", "alínea", "alinea",
    "caput", "lei", "decreto", "portaria", "resolução", "resolucao",
    "norma", "nº", "n°", "no", "n.", "numero",
    "dispõe", "dispoe", "estabelece", "considerando",
    "conforme", "respectivo", "respectiva", "respectivos", "respectivas",
    "federal", "estadual", "municipal", "união", "uniao", "nacional",
    "capítulo", "capitulo", "seção", "secao", "título", "titulo",

    # Estrutura de documento
    "processo", "documento", "anexo", "anexos", "folha", "fls", "fl",
    "item", "itens", "figura", "tabela", "capitulo", "capítulos",

    # Conectores
    "assim", "dessa", "desse", "daquele", "daquela", "daqueles", "daquelas",
    "demais", "mesmo", "mesma", "mesmos", "mesmas",

    # Termos genéricos comuns aos dois tipos
    "ambient", "áre", "águ", "local", "pont", "realiz", "tod", "outr",
    "sobr", "form", "pod", "dev", "mai", "total",

    # Termos ambientais genéricos
    "espéci", "mei", "program", "ativ", "projet",
    "empreend", "reg", "ocorr",

    # Ruído TF-IDF
    "brasil", "geral", "municip", "sant", "quadr",

    # Estruturais
    "apresent", "camp", "registr", "part", "unidade",
}

stopwords_total = stopwords_base.union(stopwords_extra)

print("Qtd stopwords padrão:", len(stopwords_base))
print("Qtd stopwords total:", len(stopwords_total))

Qtd stopwords padrão: 207
Qtd stopwords total: 301


[nltk_data] Downloading package punkt to /home/wpm/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/wpm/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/wpm/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /home/wpm/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


## 4. Função de limpeza (limpar_texto)

In [4]:
stemmer = RSLPStemmer()

def limpar_texto(texto: str) -> str:
    """
    Limpeza básica + stemming:
    - minúsculo
    - remove números
    - remove pontuação
    - tokeniza
    - remove stopwords e tokens curtos
    - aplica stemmer RSLP
    """
    if pd.isna(texto):
        return ""

    texto = texto.lower()
    texto = re.sub(r"\d+", " ", texto)
    texto = re.sub(r"[^\w\s]", " ", texto)

    tokens = word_tokenize(texto, language="portuguese")

    tokens = [
        t for t in tokens
        if t not in stopwords_total and len(t) > 2
    ]

    tokens = [stemmer.stem(t) for t in tokens]

    return " ".join(tokens)

# Teste rápido
print(limpar_texto(df["texto"].iloc[0])[:300])

mmx min rio miner logís ltd min geral rio jan eia instal oper minerodut doc mmx min rio miner logís ltd min geral rio jan estud impact ambient instal oper minerodut julh mmx min rio miner logís ltd min geral rio jan eia instal oper minerodut doc índic volum empreend equip técn dad empreend caracter 


## 5. Aplicar limpeza a todo o dataset

In [5]:
df["texto_limpo"] = (df["arquivo"].astype(str) + " " + df["texto"].astype(str)).apply(limpar_texto)

df[["arquivo", "classe", "texto_limpo"]].head()

,arquivo,classe,texto_limpo
0,01-0085-06-A-001 EIA FINAL V2.PDF,1,eia final pdf mmx min rio miner logís ltd min ...
1,0643840 - Estudo Ambiental TOMO I e II _064384...,1,estud ambient tom pdf dnit sum tom introduç in...
2,07__reas de Influ_ncia do Projeto.pdf,1,__re influ_nc projet pdf áre influ projet áre ...
3,08_Diagn_stico Ambiental_MF_Parte 01 de 03.pdf,1,_diagn_s ambiental_mf_part pdf diagnóst ambien...
4,09 - Medidas e Programas.pdf,1,med program pdf estud impact ambient med progr...


## 6. Separar treino e teste

In [6]:
X = df["texto_limpo"].values
y = df["classe"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")
print(f"EIA no treino: {y_train.sum()} | EIA no teste: {y_test.sum()}")

Treino: 124 amostras | Teste: 31 amostras
EIA no treino: 61 | EIA no teste: 15


## 7. Função auxiliar para avaliar modelos

**Mudança v4:** cross-validation agora reporta `accuracy`, `f1` e `recall` para a Classe 1 (EIA). Acurácia sozinha é enganosa com classes desbalanceadas.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def avaliar_modelo(nome, modelo_base):
    """
    Monta pipeline TF-IDF + modelo,
    roda cross-validation estratificada (5-fold) com múltiplas métricas
    e avalia no conjunto de teste.
    """
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.90,
            sublinear_tf=True,
        )),
        ("clf", modelo_base),
    ])

    # Cross-validation com múltiplas métricas
    cv_results = cross_validate(
        pipeline, X, y, cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_eia": "f1",          # F1 da classe positiva (EIA)
            "recall_eia": "recall",  # Recall da classe positiva (EIA)
        },
        return_train_score=False
    )

    print(f"\n=== {nome} — Cross-validation (5-fold) ===")
    print(f"  Acurácia : {np.round(cv_results['test_accuracy'], 3)}  → média {cv_results['test_accuracy'].mean():.3f}")
    print(f"  F1 (EIA) : {np.round(cv_results['test_f1_eia'], 3)}  → média {cv_results['test_f1_eia'].mean():.3f}")
    print(f"  Recall EIA: {np.round(cv_results['test_recall_eia'], 3)}  → média {cv_results['test_recall_eia'].mean():.3f}")

    # Avalia no conjunto de teste
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"\n=== {nome} — Desempenho no conjunto de teste ===")
    print(classification_report(y_test, y_pred, digits=3))
    print("Matriz de confusão:")
    print(confusion_matrix(y_test, y_pred))

    return pipeline, cv_results['test_f1_eia'].mean()

## 8. Baseline 1: Logistic Regression

In [8]:
modelo_lr = LogisticRegression(
    max_iter=3000,
    n_jobs=-1,
    class_weight="balanced",
    solver="lbfgs"
)

pipeline_lr, f1_lr = avaliar_modelo("Logistic Regression", modelo_lr)


=== Logistic Regression — Cross-validation (5-fold) ===
  Acurácia : [0.871 0.742 0.806 0.839 0.871]  → média 0.826
  F1 (EIA) : [0.867 0.733 0.8   0.839 0.882]  → média 0.824
  Recall EIA: [0.812 0.733 0.8   0.867 1.   ]  → média 0.843

=== Logistic Regression — Desempenho no conjunto de teste ===
              precision    recall  f1-score   support

           0      0.929     0.812     0.867        16
           1      0.824     0.933     0.875        15

    accuracy                          0.871        31
   macro avg      0.876     0.873     0.871        31
weighted avg      0.878     0.871     0.871        31

Matriz de confusão:
[[13  3]
 [ 1 14]]


## 9. Baseline 2: SGDClassifier

In [9]:
modelo_sgd = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

pipeline_sgd, f1_sgd = avaliar_modelo("SGDClassifier", modelo_sgd)


=== SGDClassifier — Cross-validation (5-fold) ===
  Acurácia : [0.903 0.806 0.903 0.903 0.806]  → média 0.865
  F1 (EIA) : [0.903 0.812 0.903 0.897 0.824]  → média 0.868
  Recall EIA: [0.875 0.867 0.933 0.867 0.933]  → média 0.895

=== SGDClassifier — Desempenho no conjunto de teste ===
              precision    recall  f1-score   support

           0      0.923     0.750     0.828        16
           1      0.778     0.933     0.848        15

    accuracy                          0.839        31
   macro avg      0.850     0.842     0.838        31
weighted avg      0.853     0.839     0.838        31

Matriz de confusão:
[[12  4]
 [ 1 14]]


## 10. Baseline 3: LinearSVC (calibrado)

**Mudança v4:** o LinearSVC não tem `predict_proba` nativo. Para usá-lo no `voting='soft'`, envolvemos com `CalibratedClassifierCV`, que aplica calibração de Platt e expõe probabilidades.

In [10]:
modelo_svc_base = LinearSVC(
    class_weight="balanced",
    max_iter=3000
)

# CalibratedClassifierCV adiciona predict_proba ao LinearSVC
modelo_svc = CalibratedClassifierCV(modelo_svc_base, cv=3)

pipeline_svc, f1_svc = avaliar_modelo("LinearSVC (calibrado)", modelo_svc)


=== LinearSVC (calibrado) — Cross-validation (5-fold) ===
  Acurácia : [0.903 0.742 0.839 0.903 0.871]  → média 0.852
  F1 (EIA) : [0.903 0.733 0.828 0.897 0.882]  → média 0.849
  Recall EIA: [0.875 0.733 0.8   0.867 1.   ]  → média 0.855

=== LinearSVC (calibrado) — Desempenho no conjunto de teste ===
              precision    recall  f1-score   support

           0      0.929     0.812     0.867        16
           1      0.824     0.933     0.875        15

    accuracy                          0.871        31
   macro avg      0.876     0.873     0.871        31
weighted avg      0.878     0.871     0.871        31

Matriz de confusão:
[[13  3]
 [ 1 14]]


## 11. Comparar médias de F1 (EIA)

In [11]:
medias_f1 = {
    "LogisticRegression": f1_lr,
    "SGDClassifier": f1_sgd,
    "LinearSVC (calibrado)": f1_svc,
}

print("=== Médias de F1 (EIA) — cross-validation ===")
for nome, valor in sorted(medias_f1.items(), key=lambda x: -x[1]):
    print(f"  {nome}: {valor:.3f}")

print("\n→ Os 3 modelos entram no ensemble com voting='soft'.")


=== Médias de F1 (EIA) — cross-validation ===
  SGDClassifier: 0.868
  LinearSVC (calibrado): 0.849
  LogisticRegression: 0.824

→ Os 3 modelos entram no ensemble com voting='soft'.
→ RandomForest foi excluído: recall de EIA era 33% isoladamente.


## 12. Ensemble v4 — VotingClassifier com `voting='soft'`

**Mudanças v4:**
- `voting='soft'`: usa probabilidades em vez de votos binários;
- RandomForest **removido** (recall EIA era 33%);
- LinearSVC **incluído** (excluído erroneamente na v3);
- `weights=[2, 2, 1]`: LR e SGD recebem peso maior que SVC por serem mais estáveis neste dataset.

In [12]:
clf1 = LogisticRegression(
    max_iter=3000,
    n_jobs=-1,
    class_weight="balanced",
    solver="lbfgs"
)

clf2 = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

clf3 = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", max_iter=3000),
    cv=3
)

ensemble = VotingClassifier(
    estimators=[
        ("lr",  clf1),
        ("sgd", clf2),
        ("svc", clf3),
    ],
    voting="soft",        # usa probabilidades
    weights=[2, 2, 1],    # LR e SGD têm mais peso
)

pipeline_ensemble = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.90,
        sublinear_tf=True,
    )),
    ("ensemble", ensemble),
])

# Cross-validation do ensemble com múltiplas métricas
cv_ens = cross_validate(
    pipeline_ensemble, X, y, cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1_eia": "f1",
        "recall_eia": "recall",
    }
)

print("=== Ensemble v4 — Cross-validation (5-fold) ===")
print(f"  Acurácia : {np.round(cv_ens['test_accuracy'], 3)}  → média {cv_ens['test_accuracy'].mean():.3f}")
print(f"  F1 (EIA) : {np.round(cv_ens['test_f1_eia'], 3)}  → média {cv_ens['test_f1_eia'].mean():.3f}")
print(f"  Recall EIA: {np.round(cv_ens['test_recall_eia'], 3)}  → média {cv_ens['test_recall_eia'].mean():.3f}")

# Avalia no conjunto de teste
pipeline_ensemble.fit(X_train, y_train)
y_pred_ens = pipeline_ensemble.predict(X_test)

print("\n=== Ensemble v4 — Desempenho no conjunto de teste ===")
print(classification_report(y_test, y_pred_ens, digits=3))
print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_ens))

=== Ensemble v4 — Cross-validation (5-fold) ===
  Acurácia : [0.903 0.774 0.871 0.903 0.806]  → média 0.852
  F1 (EIA) : [0.903 0.774 0.867 0.897 0.824]  → média 0.853
  Recall EIA: [0.875 0.8   0.867 0.867 0.933]  → média 0.868

=== Ensemble v4 — Desempenho no conjunto de teste ===
              precision    recall  f1-score   support

           0      1.000     0.750     0.857        16
           1      0.789     1.000     0.882        15

    accuracy                          0.871        31
   macro avg      0.895     0.875     0.870        31
weighted avg      0.898     0.871     0.869        31

Matriz de confusão:
[[12  4]
 [ 0 15]]


## 13. Threshold tuning — ajustar limiar de decisão

Com `voting='soft'` temos probabilidades. O limiar padrão é 0.5 (50%), mas podemos baixá-lo para capturar mais EIAs (aumentar recall) ao custo de aceitar alguns falsos positivos.

Esta célula encontra o **menor threshold que mantém precisão de EIA ≥ 0.85**.

In [13]:
# Probabilidades da classe 1 (EIA) no conjunto de teste
y_proba = pipeline_ensemble.predict_proba(X_test)[:, 1]

# Curva precision-recall para diferentes thresholds
precisoes, recalls, thresholds = precision_recall_curve(y_test, y_proba)

print("Threshold | Precisão EIA | Recall EIA | F1 EIA")
print("-" * 50)
for t in np.arange(0.30, 0.71, 0.05):
    y_pred_t = (y_proba >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    marker = " ← padrão" if abs(t - 0.50) < 0.01 else ""
    print(f"  {t:.2f}    |   {p:.3f}      |   {r:.3f}    | {f1:.3f}{marker}")

# Escolher o melhor threshold: maior F1 com precisão >= 0.85
melhor_t = 0.50
melhor_f1 = 0.0
for t in np.arange(0.25, 0.75, 0.01):
    y_pred_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    if p >= 0.85 and f1 > melhor_f1:
        melhor_f1 = f1
        melhor_t = t

print(f"\n→ Melhor threshold (precisão ≥ 0.85): {melhor_t:.2f}  —  F1: {melhor_f1:.3f}")

Threshold | Precisão EIA | Recall EIA | F1 EIA
--------------------------------------------------
  0.30    |   0.600      |   1.000    | 0.750
  0.35    |   0.625      |   1.000    | 0.769
  0.40    |   0.652      |   1.000    | 0.789
  0.45    |   0.789      |   1.000    | 0.882
  0.50    |   0.789      |   1.000    | 0.882 ← padrão
  0.55    |   0.812      |   0.867    | 0.839
  0.60    |   0.867      |   0.867    | 0.867
  0.65    |   0.867      |   0.867    | 0.867
  0.70    |   0.917      |   0.733    | 0.815

→ Melhor threshold (precisão ≥ 0.85): 0.59  —  F1: 0.867


## 14. Avaliação final com threshold otimizado

In [15]:
THRESHOLD_FINAL = melhor_t  # use o valor encontrado acima, ou defina manualmente

y_pred_final = (y_proba >= THRESHOLD_FINAL).astype(int)

print(f"=== Ensemble v4 — threshold={THRESHOLD_FINAL:.2f} ===")
print(classification_report(y_test, y_pred_final, digits=3))
print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_final))

print(f"\nROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

=== Ensemble v4 — threshold=0.59 ===
              precision    recall  f1-score   support

           0      0.875     0.875     0.875        16
           1      0.867     0.867     0.867        15

    accuracy                          0.871        31
   macro avg      0.871     0.871     0.871        31
weighted avg      0.871     0.871     0.871        31

Matriz de confusão:
[[14  2]
 [ 2 13]]

ROC-AUC: 0.954


## 15. Treinar em 100% dos dados + Validação cruzada final

**Mudança v4 (crítica):** Na v3, o modelo era salvo sem nenhuma avaliação após o `fit(X, y)`. Aqui fazemos uma validação cruzada estratificada final **antes** de salvar, garantindo que as métricas reflitam o modelo de produção.

In [16]:
# Recriar o pipeline do zero (estado limpo) para treinar em 100%
clf1_prod = LogisticRegression(
    max_iter=3000, n_jobs=-1, class_weight="balanced", solver="lbfgs"
)
clf2_prod = SGDClassifier(
    loss="log_loss", max_iter=2000, class_weight="balanced",
    n_jobs=-1, random_state=42
)
clf3_prod = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", max_iter=3000), cv=3
)

ensemble_prod = VotingClassifier(
    estimators=[("lr", clf1_prod), ("sgd", clf2_prod), ("svc", clf3_prod)],
    voting="soft",
    weights=[2, 2, 1],
)

pipeline_producao = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3), min_df=2, max_df=0.90, sublinear_tf=True,
    )),
    ("ensemble", ensemble_prod),
])

# ── Validação cruzada final em TODO o dataset ──────────────────────────────
print("Rodando validação cruzada final em 100% dos dados...")
cv_final = cross_validate(
    pipeline_producao, X, y, cv=cv,
    scoring={
        "accuracy":   "accuracy",
        "f1_eia":     "f1",
        "recall_eia": "recall",
        "roc_auc":    "roc_auc",
    }
)

print("\n=== Validação cruzada do modelo de produção (5-fold, 100% dos dados) ===")
print(f"  Acurácia  : {np.round(cv_final['test_accuracy'],   3)}  → média {cv_final['test_accuracy'].mean():.3f} ± {cv_final['test_accuracy'].std():.3f}")
print(f"  F1 (EIA)  : {np.round(cv_final['test_f1_eia'],     3)}  → média {cv_final['test_f1_eia'].mean():.3f} ± {cv_final['test_f1_eia'].std():.3f}")
print(f"  Recall EIA: {np.round(cv_final['test_recall_eia'], 3)}  → média {cv_final['test_recall_eia'].mean():.3f} ± {cv_final['test_recall_eia'].std():.3f}")
print(f"  ROC-AUC   : {np.round(cv_final['test_roc_auc'],    3)}  → média {cv_final['test_roc_auc'].mean():.3f} ± {cv_final['test_roc_auc'].std():.3f}")

# ── Treinar em 100% e salvar ───────────────────────────────────────────────
print("\nTreinando em 100% dos dados para produção...")
pipeline_producao.fit(X, y)

joblib.dump(pipeline_producao, "modelo_eia_vs_outro_ensemble_v4.joblib")
print("Modelo salvo: modelo_eia_vs_outro_ensemble_v4.joblib")
print(f"\nResumo do modelo salvo:")
print(f"  - Voting: soft, weights=[2, 2, 1]")
print(f"  - Estimadores: LR, SGD, LinearSVC(calibrado)")
print(f"  - TF-IDF: ngram (1,3), min_df=2, max_df=0.90, sublinear_tf")
print(f"  - F1 médio EIA (CV final): {cv_final['test_f1_eia'].mean():.3f}")
print(f"  - Recall médio EIA (CV final): {cv_final['test_recall_eia'].mean():.3f}")

Rodando validação cruzada final em 100% dos dados...

=== Validação cruzada do modelo de produção (5-fold, 100% dos dados) ===
  Acurácia  : [0.903 0.774 0.871 0.903 0.806]  → média 0.852 ± 0.052
  F1 (EIA)  : [0.903 0.774 0.867 0.897 0.824]  → média 0.853 ± 0.048
  Recall EIA: [0.875 0.8   0.867 0.867 0.933]  → média 0.868 ± 0.042
  ROC-AUC   : [0.971 0.875 0.946 0.929 0.933]  → média 0.931 ± 0.031

Treinando em 100% dos dados para produção...
Modelo salvo: modelo_eia_vs_outro_ensemble_v4.joblib

Resumo do modelo salvo:
  - Voting: soft, weights=[2, 2, 1]
  - Estimadores: LR, SGD, LinearSVC(calibrado)
  - TF-IDF: ngram (1,3), min_df=2, max_df=0.90, sublinear_tf
  - F1 médio EIA (CV final): 0.853
  - Recall médio EIA (CV final): 0.868


## 16. Função para classificar novos documentos (PDF / TXT)

**Mudança v4:** a função aceita um `threshold` opcional. Se não passado, usa o `THRESHOLD_FINAL` encontrado no tuning.

In [17]:
import pdfplumber
import os

modelo_final = joblib.load("modelo_eia_vs_outro_ensemble_v4.joblib")

def extrair_texto_arquivo(caminho):
    caminho = str(caminho)

    if caminho.lower().endswith(".pdf"):
        try:
            with pdfplumber.open(caminho) as pdf:
                paginas = [p.extract_text() or "" for p in pdf.pages]
                return "\n".join(paginas)
        except Exception as e:
            print("Erro ao ler PDF:", e)
            return ""

    elif caminho.lower().endswith(".txt"):
        try:
            with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
        except Exception as e:
            print("Erro ao ler TXT:", e)
            return ""

    else:
        raise ValueError("Formato não reconhecido. Use PDF ou TXT.")


def classificar_documento(caminho_arquivo, threshold=None):
    """
    Classifica um documento PDF ou TXT como EIA ou OUTRO.

    Parâmetros
    ----------
    caminho_arquivo : str — caminho para o arquivo
    threshold : float ou None — limiar de decisão (padrão: THRESHOLD_FINAL)

    Retorna
    -------
    dict com 'classe', 'probabilidade_eia' e 'threshold_usado'
    """
    t = threshold if threshold is not None else THRESHOLD_FINAL

    texto = extrair_texto_arquivo(caminho_arquivo)
    texto_limpo = limpar_texto(texto)

    proba_eia = modelo_final.predict_proba([texto_limpo])[0][1]
    classe = "EIA" if proba_eia >= t else "OUTRO"

    return {
        "classe": classe,
        "probabilidade_eia": round(float(proba_eia), 4),
        "threshold_usado": round(t, 2),
    }


#===== TESTE =====
arquivo_teste = "/home/wpm/Downloads/Volume_1__EIA.pdf"
resultado = classificar_documento(arquivo_teste)
print(resultado)
print("Função classificar_documento carregada. Pronta para uso.")

{'classe': 'EIA', 'probabilidade_eia': 0.7853, 'threshold_usado': np.float64(0.59)}
Função classificar_documento carregada. Pronta para uso.


In [35]:
texto = extrair_texto_arquivo("/home/wpm/Downloads/Volume_1__EIA.pdf")
print(f"Caracteres extraídos: {len(texto)}")
print(texto[:500])

Caracteres extraídos: 284225
NAÇÃO: BRASIL
ESTADO FEDERAL: CEARÁ
Prefeitura de Caucaia
Localidade "Iparana-Icarai-Parazinho"
PARQUE EÓLICO OFFSHORE CAUCAIA PARAZINHO - IPARANA
N. 59 AEROGERADORES
Estudo de Impacto Ambiental (EIA)
VOLUME 1 – Quadro programático e projetual
SEDE LEGAL BI ENERGIA LTDA AV. DESEMBARGADOR MOREIRA, 2120 SEDE LEGAL TENPROJECT SRL SAN GIORGIO DEL SANNIO
SALA 907, ALDEOTA - FORTALEZA - CE RUA DE GASPERI 61, CEP 82018
CEP 60170-002 PROVÍNCIA DE BENEVENTO, ITÁLIA
COMPANHIA COM SISTEMA GESTÃO QUALIDADE

